In [33]:
import torch
import importlib

import hdmm_utils_torch
import hdmm_torch

In [4]:
importlib.reload(hdmm_utils_torch)
from hdmm_utils_torch import mix_weights
# ---------- TEST CASES ----------
print("==== Test 1: Simple deterministic case ====")
beta = torch.tensor([[0.3, 0.5, 0.7]])
weights = mix_weights(beta)
print("beta:", beta)
print("weights:", weights)
print("sum(weights):", weights.sum(dim=-1))
# Expected: [0.3, 0.35, 0.245] → sum ≈ 0.895


print("\n==== Test 2: Batch of betas ====")
beta = torch.tensor([
    [0.1, 0.2, 1.0],
    [0.3, 0.3, 1.0],
])
weights = mix_weights(beta)
print("beta:\n", beta)
print("weights:\n", weights)
print("sum(weights):", weights.sum(dim=-1))


print("\n==== Test 3: Random beta in higher dimensions ====")
torch.manual_seed(0)
beta = torch.rand(2, 3, 5)
weights = mix_weights(beta)
print("beta shape:", beta.shape)
print("weights shape:", weights.shape)
print("Each row sum (should < 1):\n", weights.sum(dim=-1))


print("\n==== Test 4: Axis check (non-last dimension) ====")
beta = torch.tensor([
    [[0.3, 0.5, 0.7]],
    [[0.2, 0.4, 0.6]]
]).permute(1, 0, 2)   # shape (1, 2, 3)
weights = mix_weights(beta, axis=1)
print("beta shape:", beta.shape)
print("weights shape:", weights.shape)
print("weights:\n", weights)

==== Test 1: Simple deterministic case ====
beta: tensor([[0.3000, 0.5000, 0.7000]])
weights: tensor([[0.3000, 0.3500, 0.2450]])
sum(weights): tensor([0.8950])

==== Test 2: Batch of betas ====
beta:
 tensor([[0.1000, 0.2000, 1.0000],
        [0.3000, 0.3000, 1.0000]])
weights:
 tensor([[0.1000, 0.1800, 0.7200],
        [0.3000, 0.2100, 0.4900]])
sum(weights): tensor([1., 1.])

==== Test 3: Random beta in higher dimensions ====
beta shape: torch.Size([2, 3, 5])
weights shape: torch.Size([2, 3, 5])
Each row sum (should < 1):
 tensor([[0.9360, 0.9961, 0.7765],
        [0.9825, 0.9988, 0.9896]])

==== Test 4: Axis check (non-last dimension) ====
beta shape: torch.Size([1, 2, 3])
weights shape: torch.Size([1, 2, 3])
weights:
 tensor([[[0.3000, 0.5000, 0.7000],
         [0.1400, 0.2000, 0.1800]]])


In [6]:
importlib.reload(hdmm_utils_torch)
from hdmm_utils_torch import suffix_sum
print("\n==== Test 5: Suffix sum ====")

x = torch.tensor([[1, 2, 3],
                  [4, 5, 6]], dtype=torch.float32)
print(suffix_sum(x))



==== Test 5: Suffix sum ====
tensor([[5.0000e+00, 3.0000e+00, 1.0000e-10],
        [1.1000e+01, 6.0000e+00, 1.0000e-10]])


In [8]:
importlib.reload(hdmm_utils_torch)
from hdmm_utils_torch import dirichlet_posterior
print("\n==== Test 6: Dirichlet posterior sampling ====")

torch.manual_seed(0)

obs = torch.tensor([[1., 0., 0.],
                    [0., 1., 0.],
                    [0., 0., 1.]])
params = torch.tensor([0.5, 0.5, 0.5])

sample = dirichlet_posterior(obs, params, scaling_constant=1.0)
print("Posterior sample:", sample)
print("Sum:", sample.sum())



==== Test 6: Dirichlet posterior sampling ====
Posterior sample: tensor([0.7543, 0.1766, 0.0691])
Sum: tensor(1.)


In [13]:
importlib.reload(hdmm_utils_torch)
from hdmm_utils_torch import nig_posterior
print("\n==== Test 7: Normal-Inverse-Gamma posterior sampling ====")

torch.manual_seed(0)

obs = torch.tensor([2.1, 1.9, 2.3, 2.0, 2.2])
params = [0.0, 1.0, 2.0, 2.0]  # [mu, kappa, alpha, beta]

new_mu, new_sigma = nig_posterior(obs, params, scale_constant=1.0)

print("new_mu:", new_mu)
print("new_sigma:", new_sigma)



==== Test 7: Normal-Inverse-Gamma posterior sampling ====
new_mu: tensor(1.6674)
new_sigma: tensor(0.4758)


In [14]:
importlib.reload(hdmm_utils_torch)
from hdmm_utils_torch import nig_posterior_batch
print("\n==== Test 8: Normal-Inverse-Gamma posterior sampling (batch) ====")
torch.manual_seed(0)

# Suppose we have K=3 regression components
count = torch.tensor([10., 15., 20.])
mean = torch.tensor([2.0, 1.5, 3.0])
sum_var = torch.tensor([1.5, 2.0, 2.5])
params = [0.0, 1.0, 2.0, 2.0]  # [mu0, kappa0, alpha0, beta0]

new_mu, new_sigma = nig_posterior_batch(count, mean, sum_var, params, scale_constant=1.0)

print("new_mu:", new_mu)
print("new_sigma:", new_sigma)



==== Test 8: Normal-Inverse-Gamma posterior sampling (batch) ====
new_mu: tensor([1.5523, 1.2695, 3.0775])
new_sigma: tensor([0.3976, 0.4881, 0.9035])


In [15]:
importlib.reload(hdmm_utils_torch)
from hdmm_utils_torch import gaussian_mixture_posterior
print("\n==== Test 9: Gaussian Mixture posterior sampling ====")
torch.manual_seed(0)

score = torch.tensor(2.0)
weight = torch.tensor([0.3, 0.5, 0.2])
components = torch.tensor([
    [1.5, 0.25],  # mean, variance
    [2.0, 0.10],
    [3.0, 0.20]
])

sample = gaussian_mixture_posterior(score, weight, components)
print("Sampled component index:", sample.item())



==== Test 9: Gaussian Mixture posterior sampling ====
Sampled component index: 1


In [17]:
importlib.reload(hdmm_utils_torch)
from hdmm_utils_torch import topic_mixture_posterior
print("\n==== Test 10: Topic Mixture posterior sampling ====")

torch.manual_seed(0)

V, K = 5, 3
word = torch.tensor([0., 0., 1., 0., 0.])  # one-hot for word 2
weight = torch.tensor([0.2, 0.5, 0.3])
components = torch.tensor([
    [0.1, 0.2, 0.3, 0.2, 0.2],
    [0.2, 0.1, 0.5, 0.1, 0.1],
    [0.3, 0.3, 0.1, 0.2, 0.1]
])

sample = topic_mixture_posterior(word, weight, components)
print("Sampled topic index:", sample.item())





==== Test 10: Topic Mixture posterior sampling ====
Sampled topic index: 1


In [18]:
importlib.reload(hdmm_utils_torch)
from hdmm_utils_torch import get_unique_rows_and_positions
print("\n==== Test 11: Get unique rows and their positions ====")

x = torch.tensor([
    [1, 2],
    [3, 4],
    [1, 2],
    [5, 6],
    [3, 4]
])

unique_rows, positions = get_unique_rows_and_positions(x)

print("Unique rows:")
print(unique_rows)
print("\nPositions (indices for each unique row):")
for i, pos in enumerate(positions):
    print(f"Row {i}: {pos.tolist()}")



==== Test 11: Get unique rows and their positions ====
Unique rows:
tensor([[1, 2],
        [3, 4],
        [5, 6]])

Positions (indices for each unique row):
Row 0: [0, 2]
Row 1: [1, 4]
Row 2: [3]


In [20]:
importlib.reload(hdmm_utils_torch)
from hdmm_utils_torch import beta_mixture_posterior
print("\n==== Test 12: Beta Mixture posterior sampling ====")

torch.manual_seed(0)

C = 4
doc_nu = torch.tensor([0.2, 0.6, 0.8, 0.9])
doc_alpha = torch.tensor([2.0, 2.0, 2.0, 2.0])
doc_beta = torch.tensor([3.0, 3.0, 3.0, 3.0])
cluster_prob = torch.tensor([0.1, 0.4, 0.3, 0.2])

new_cat, prob = beta_mixture_posterior(doc_nu, [doc_alpha, doc_beta], cluster_prob)
print("Sampled category:", new_cat.item())
print("Posterior probabilities:", prob)



==== Test 12: Beta Mixture posterior sampling ====
Sampled category: 2
Posterior probabilities: tensor([0.1000, 0.4000, 0.3000, 0.2000])


In [24]:
importlib.reload(hdmm_utils_torch)
from hdmm_utils_torch import gather_middle_slice
print("\n==== Test 13: Gather middle slice from tensor ====")

# Suppose x has shape (2, 3, 4, 5)
x = torch.arange(2*3*4*5).reshape(2, 3, 4, 5)

# Select idx = [1, 2] (i.e. D1=1, D2=2)
idx = torch.tensor([1, 2])

result = gather_middle_slice(x, idx)

print("Input")
print(x)
print("Input shape:", x.shape)
print("Result shape:", result.shape)
print("Result:")
print(result)



==== Test 13: Gather middle slice from tensor ====
Input
tensor([[[[  0,   1,   2,   3,   4],
          [  5,   6,   7,   8,   9],
          [ 10,  11,  12,  13,  14],
          [ 15,  16,  17,  18,  19]],

         [[ 20,  21,  22,  23,  24],
          [ 25,  26,  27,  28,  29],
          [ 30,  31,  32,  33,  34],
          [ 35,  36,  37,  38,  39]],

         [[ 40,  41,  42,  43,  44],
          [ 45,  46,  47,  48,  49],
          [ 50,  51,  52,  53,  54],
          [ 55,  56,  57,  58,  59]]],


        [[[ 60,  61,  62,  63,  64],
          [ 65,  66,  67,  68,  69],
          [ 70,  71,  72,  73,  74],
          [ 75,  76,  77,  78,  79]],

         [[ 80,  81,  82,  83,  84],
          [ 85,  86,  87,  88,  89],
          [ 90,  91,  92,  93,  94],
          [ 95,  96,  97,  98,  99]],

         [[100, 101, 102, 103, 104],
          [105, 106, 107, 108, 109],
          [110, 111, 112, 113, 114],
          [115, 116, 117, 118, 119]]]])
Input shape: torch.Size([2, 3, 4, 5])
R

In [28]:
importlib.reload(hdmm_utils_torch)
from hdmm_utils_torch import partial_index
print("\n==== Test 14: Partial Indexing with Different Modes ====")

x = torch.arange(2*3*4*5).reshape(2, 3, 4, 5)

# Example 1: exact index
out1 = partial_index(x, [1, 2])
print("out1 shape:", out1.shape)
print("out1 (first row):", out1[0])

# Example 2: clipping
out2 = partial_index(x, [10, -1], mode="clip")
print("out2 (clipped):", out2[0])

# Example 3: wrapping
out3 = partial_index(x, [4, 5], mode="wrap")
print("out3 (wrapped):", out3[0])



==== Test 14: Partial Indexing with Different Modes ====
out1 shape: torch.Size([4, 5])
out1 (first row): tensor([100, 101, 102, 103, 104])
out2 (clipped): tensor([60, 61, 62, 63, 64])
out3 (wrapped): tensor([40, 41, 42, 43, 44])


In [32]:
importlib.reload(hdmm_utils_torch)
from hdmm_utils_torch import set_by_multi_index
print("\n==== Test 15: Set by Multi-Index ====")

x = torch.zeros((2, 3, 4, 5))
idx = [1, 2]
value = torch.arange(5).float()

out = set_by_multi_index(x, idx, value)

print("Updated tensor slice at [1,2]:")
print(out[1, 2])



==== Test 15: Set by Multi-Index ====
Updated tensor slice at [1,2]:
tensor([[0., 1., 2., 3., 4.],
        [0., 1., 2., 3., 4.],
        [0., 1., 2., 3., 4.],
        [0., 1., 2., 3., 4.]])


In [34]:
import torch

def advanced_multi_index_select(a: torch.Tensor, b: torch.Tensor, dims: torch.Tensor):
    """
    Generalized multi-dimensional indexing.

    Args:
        a: tensor of shape (D0, D1, ..., Dp)
        b: LongTensor of shape (N, n)
           where each row gives indices along the dimensions specified in `dims`
        dims: 1D LongTensor or list of ints of length n
           specifying which dimensions of `a` are being indexed.

    Returns:
        Tensor of shape (N, remaining_dims_of_a)
        where dimensions not in `dims` are kept.
    """
    assert b.shape[1] == len(dims), "b.shape[1] must match len(dims)"

    dims = torch.as_tensor(dims, dtype=torch.long, device=b.device)
    N, n = b.shape
    total_dims = a.ndim

    # 1️⃣ Move indexed dims to the front so we can flatten them easily
    permute_order = torch.cat([dims, torch.tensor([d for d in range(total_dims) if d not in dims], device=b.device)])
    a_perm = a.permute(*permute_order)
    
    prefix_shape = [a.shape[d] for d in dims]
    suffix_shape = [a.shape[d] for d in range(total_dims) if d not in dims]
    flat_a = a_perm.reshape(int(torch.prod(torch.tensor(prefix_shape))), *suffix_shape)

    # 2️⃣ Compute flat indices corresponding to b[:, dims]
    strides = torch.tensor(
        [int(torch.prod(torch.tensor(prefix_shape[i+1:]))) for i in range(n)],
        device=b.device
    )
    flat_idx = (b * strides).sum(dim=1)

    # 3️⃣ Gather the indexed rows
    result = flat_a[flat_idx]
    return result


In [35]:
a = torch.arange(2*3*4).reshape(2, 3, 4)
b = torch.tensor([[0, 1], [1, 2]])   # index (s, c)
dims = [0, 1]

out = advanced_multi_index_select(a, b, dims)
print(out)
print(out.shape)


tensor([[ 4,  5,  6,  7],
        [20, 21, 22, 23]])
torch.Size([2, 4])


In [5]:
import torch

def safe_update_scatter(tensor: torch.Tensor,
                        indices: torch.LongTensor,
                        values: torch.Tensor) -> torch.Tensor:
    """
    Vectorized autograd-safe update along the last dimension using torch.scatter.

    tensor: (..., k)
    indices: (num_updates, tensor.ndim - 1)
              Each row gives the coordinates (e.g. (a, b)).
    values:  (num_updates, k)
              Replacement rows.
    """
    out = tensor.clone()
    *prefix_shape, k = out.shape
    n_prefix = int(torch.prod(torch.tensor(prefix_shape, device=tensor.device)))

    flat = out.view(n_prefix, k)

    # Compute strides for flattening all but last dimension
    strides = []
    for i in range(len(prefix_shape)):
        if i + 1 < len(prefix_shape):
            stride = int(torch.prod(torch.tensor(prefix_shape[i+1:], device=tensor.device)))
        else:
            stride = 1
        strides.append(stride)
    strides = torch.tensor(strides, device=tensor.device, dtype=torch.long)

    # Compute flattened indices
    flat_idx = (indices.long() * strides).sum(dim=1)

    # Expand flat indices to match (num_updates, k)
    scatter_index = flat_idx.unsqueeze(1).expand(-1, k)

    # Scatter new values into the flattened view
    flat = flat.scatter(0, scatter_index, values)

    return flat.view(out.shape)


In [7]:
X = torch.zeros(5, 4, 3, requires_grad=True)
indices = torch.tensor([[2, 1]])        # (a=2, b=1)
new_value = torch.tensor([[1., 2., 3.]], requires_grad=True)

Y = safe_update_scatter(X, indices, new_value)

print(Y)   # tensor([1., 2., 3.])


tensor([[[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [1., 2., 3.],
         [0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]]], grad_fn=<ViewBackward0>)


In [ ]:
def get_unique_rows_and_positions(x: torch.Tensor):
    """
    Get all unique rows in data and their positions.

    Args:
        x: (N, D) tensor of data points.

    Returns:
        unique_rows: (N_unique, D) tensor of unique rows.
        positions: list of index tensors; positions[i] contains the indices in x
                   that correspond to unique_rows[i].
    """
    # Ensure float32 or int dtype, no gradients
    x = x.detach()

    # Step 1: Get unique rows and inverse mapping
    unique_rows, inv_idx = torch.unique(x, dim=0, return_inverse=True)

    # Step 2: Group positions for each unique row
    positions = [(inv_idx == i).nonzero(as_tuple=False).flatten() for i in range(unique_rows.size(0))]

    return unique_rows, positions